# Multimodal RAG on a PDF Report

Extracts text, tables, and images/charts from a PDF, indexes everything in a
local vector store, and answers questions using open-source models .



In [ ]:
# 1. Install packages 
%pip install -q \
    "numpy<2.3" "scipy<1.15" "scikit-learn<1.6" \
    pymupdf sentence-transformers chromadb pandas tabulate pillow \
    qwen-vl-utils accelerate

In [1]:
# 2. Imports
import os
import io
import base64
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
import torch
from PIL import Image

import chromadb
from sentence_transformers import SentenceTransformer

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Torch CUDA available:", torch.cuda.is_available())

Torch CUDA available: True


In [2]:
# 3. Configuration
PDF_PATH = Path("/kaggle/input/datasets/sihamaitbaziz/ooredoo-rag-pdf/2025 Ooredoo Annual Report_English_compressed.pdf")

if not PDF_PATH.exists():
    local_fallback = Path("data") / PDF_PATH.name
    if local_fallback.exists():
        PDF_PATH = local_fallback

assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH} — update PDF_PATH above."

VLM_MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"   
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

IMAGE_DIR = Path("extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

CHROMA_DIR = Path("chroma_db")
COLLECTION_NAME = "report_multimodal_rag"

print("PDF:", PDF_PATH)

PDF: /kaggle/input/datasets/sihamaitbaziz/ooredoo-rag-pdf/2025 Ooredoo Annual Report_English_compressed.pdf


In [3]:
# 4. Load the vision-language model once (used for both captioning and answering)
device = "cuda" if torch.cuda.is_available() else "cpu"

vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)

if device == "cpu":
    vlm_model.to(device)

print("Vision-language model ready on:", device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vision-language model ready on: cuda


In [4]:

def run_vlm(prompt_text, image_paths=None, max_new_tokens=500):
    """Send a text prompt (optionally with images) to Qwen2-VL and return the reply text."""
    content = []
    if image_paths:
        for p in image_paths:
            content.append({"type": "image", "image": str(p)})
    content.append({"type": "text", "text": prompt_text})

    messages = [{"role": "user", "content": content}]

    text = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vlm_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(vlm_model.device)

    with torch.no_grad():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = vlm_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0].strip()

In [5]:

def summarize_visual(image_path, page_number):
    prompt = f"""This visual was extracted from page {page_number} of a PDF report.

Describe the useful information it contains.
- If it is a chart or graph: mention key values, the highest/lowest points, and the main trend.
- If it is a diagram: identify the components and how they relate.
- If it is a photo/illustration: describe the useful factual content.

Be concise and factual."""
    return run_vlm(prompt, image_paths=[image_path], max_new_tokens=350)

In [6]:

def extract_multimodal_chunks(pdf_path):
    pdf = fitz.open(str(pdf_path))
    chunks = []          # list of dicts: {text, page, modality, image_path?}
    seen_xrefs = set()

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1
        print(f"Processing page {page_number}/{len(pdf)}...")

        # --- text ---
        text = page.get_text("text").strip()
        if text:
            chunks.append({"text": text, "page": page_number, "modality": "text"})

        # --- tables ---
        try:
            for i, table in enumerate(page.find_tables().tables, start=1):
                df = table.to_pandas()
                if not df.empty:
                    chunks.append({
                        "text": df.to_markdown(index=False),
                        "page": page_number,
                        "modality": "table",
                    })
        except Exception as e:
            print("  table extraction warning:", e)

        # --- images / charts ---
        for img_num, img_info in enumerate(page.get_images(full=True), start=1):
            xref = img_info[0]
            if xref in seen_xrefs:
                continue
            seen_xrefs.add(xref)

            img_data = pdf.extract_image(xref)
            img_path = IMAGE_DIR / f"page_{page_number}_img_{img_num}.{img_data['ext']}"
            img_path.write_bytes(img_data["image"])

            # skip tiny images (logos, icons, decorative lines)
            try:
                with Image.open(img_path) as im:
                    if im.width < 120 or im.height < 120:
                        continue
            except Exception:
                continue

            try:
                summary = summarize_visual(img_path, page_number)
                chunks.append({
                    "text": summary,
                    "page": page_number,
                    "modality": "visual",
                    "image_path": str(img_path),
                })
            except Exception as e:
                print(f"  vision warning on page {page_number}:", e)

    pdf.close()
    return chunks


chunks = extract_multimodal_chunks(PDF_PATH)
print()
print("Total chunks:", len(chunks))
pd.DataFrame([{"page": c["page"], "modality": c["modality"], "preview": c["text"][:100]} for c in chunks])

Processing page 1/80...
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
  vision warning on page 1: CUDA out of memory. Tried to allocate 11.91 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.08 GiB is free. Including non-PyTorch memory, this process has 4.48 GiB memory in use. Of the allocated memory 4.03 GiB is allocated by PyTorch, and 325.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Processing page 2/80...
Processing page 3/80...
  vision warning on page 3: CUDA out of memory. Tried to allocate 11.97 GiB. GPU 0 has a total capacity of 14.56 GiB of which 9.47 GiB is free. Including non-PyTorch memory, this process has 5.09 GiB memory in use. Of the allocated memory 4.04 GiB is allocated by PyTorch, a

,page,modality,preview
0,1,text,Annual Report 2025 \nA NEW ERA \nOF GROWTH
1,2,text,Amir of the State of Qatar\nHis Highness Sheik...
2,2,visual,The image is a photograph of a person wearing ...
3,3,text,Introduction\nHighlights of 2025\n4\nChairman’...
4,4,text,Financial\nEBITDA = Revenue – Operating expens...
...,...,...,...
179,75,text,NOTES\nNotes to the Consolidated Financial Sta...
180,76,text,NOTES\nNotes to the Consolidated Financial Sta...
181,77,text,The following tables present revenue and profi...
182,78,text,i.\t Inter-segment revenues are eliminated on ...


In [7]:
# 8. Embeddings + local ChromaDB vector store
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# start fresh each run
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(COLLECTION_NAME)

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()

metadatas = [
    {"page": c["page"], "modality": c["modality"], "image_path": c.get("image_path", "")}
    for c in chunks
]
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
print("Indexed", collection.count(), "chunks in ChromaDB.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Indexed 184 chunks in ChromaDB.


In [8]:
# 9. Retriever
def retrieve(question, k=5):
    q_emb = embedder.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)

    hits = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        hits.append({"text": doc, **meta})
    return hits

In [9]:
# 10. Full RAG pipeline
def format_context(hits):
    return "\n\n".join(
        f"[Page {h['page']} | {h['modality'].upper()}]\n{h['text']}" for h in hits
    )


def ask_rag(question, k=5, show_sources=True):
    hits = retrieve(question, k=k)
    context = format_context(hits)

    image_paths = [h["image_path"] for h in hits if h.get("image_path")][:3]

    prompt = f"""You are answering questions about a PDF report.
Use ONLY the context below (and attached images, if any).
If the answer isn't there, say you could not find it. Mention page numbers.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

    answer = run_vlm(prompt, image_paths=image_paths if image_paths else None, max_new_tokens=500)

    print("QUESTION:", question)
    print()
    print("ANSWER:", answer)

    if show_sources:
        print()
        print("SOURCES:")
        for h in hits:
            print(f"  - page {h['page']} | {h['modality']}")

    return {"answer": answer, "hits": hits}

## Try it

Adjust these to match your own report's content.

In [10]:
ask_rag("What is this report about, and who is it for?")

QUESTION: What is this report about, and who is it for?

ANSWER: The report is about a new era of growth, and it is likely intended for a specific audience, such as investors, stakeholders, or employees. The report may contain information about the company's financial performance, strategic goals, and future prospects.

SOURCES:
  - page 1 | text
  - page 24 | visual
  - page 24 | visual
  - page 23 | visual
  - page 24 | visual


{'answer': "The report is about a new era of growth, and it is likely intended for a specific audience, such as investors, stakeholders, or employees. The report may contain information about the company's financial performance, strategic goals, and future prospects.",
 'hits': [{'text': 'Annual Report 2025 \nA NEW ERA \nOF GROWTH',
   'image_path': '',
   'page': 1,
   'modality': 'text'},
  {'text': "The image is a red circular icon with a white silhouette of a person's hands holding a person's head. This icon is likely used to represent care, support, or protection. Without additional context from the PDF report, it is not possible to provide specific values, trends, or components related to the icon. The icon itself does not contain any data or information that can be quantified or analyzed.",
   'image_path': 'extracted_images/page_24_img_5.png',
   'page': 24,
   'modality': 'visual'},
  {'text': 'The image is a red circular icon with a white shield symbol and a checkmark inside 

In [11]:
ask_rag("Summarize the key financial or performance figures mentioned in the report.")

QUESTION: Summarize the key financial or performance figures mentioned in the report.

ANSWER: The key financial or performance figures mentioned in the report include:

- Revenue: 2,540 million QR in 2021, 2,809 million QR in 2022, 2,914 million QR in 2023, 3,132 million QR in 2024, and 3,253 million QR in 2025.
- EBITDA: 738 million QR in 2021, 850 million QR in 2022, 971 million QR in 2023, 839 million QR in 2024, and 1,067 million QR in 2025.
- EBITDA margin: 29% in 2021, 30% in 2022, 33% in 2023, 27% in 2024, and 33% in 2025.
- Blended ARPU: 66.0 QR in 2021, 61.8 QR in 2022, 60.8 QR in 2023, 62.4 QR in 2024, and 65.5 QR in 2025.
- Employees: 1,230 in 2021, 1,423 in 2022, 1,349 in 2023, 1,360 in 2024, and 1,366 in 2025.

These figures provide a snapshot of the financial performance and key metrics of the company over the past five years.

SOURCES:
  - page 13 | table
  - page 12 | table
  - page 16 | table
  - page 14 | table
  - page 40 | text


{'answer': 'The key financial or performance figures mentioned in the report include:\n\n- Revenue: 2,540 million QR in 2021, 2,809 million QR in 2022, 2,914 million QR in 2023, 3,132 million QR in 2024, and 3,253 million QR in 2025.\n- EBITDA: 738 million QR in 2021, 850 million QR in 2022, 971 million QR in 2023, 839 million QR in 2024, and 1,067 million QR in 2025.\n- EBITDA margin: 29% in 2021, 30% in 2022, 33% in 2023, 27% in 2024, and 33% in 2025.\n- Blended ARPU: 66.0 QR in 2021, 61.8 QR in 2022, 60.8 QR in 2023, 62.4 QR in 2024, and 65.5 QR in 2025.\n- Employees: 1,230 in 2021, 1,423 in 2022, 1,349 in 2023, 1,360 in 2024, and 1,366 in 2025.\n\nThese figures provide a snapshot of the financial performance and key metrics of the company over the past five years.',
 'hits': [{'text': '| Financial performance                       | Col1   |\n|:--------------------------------------------|:-------|\n| 2021 2022 2023 2024 2025                    |        |\n| Revenue QR millions 2,5

In [12]:
ask_rag("Are there any charts or diagrams in the report? What do they show?")

QUESTION: Are there any charts or diagrams in the report? What do they show?

ANSWER: Yes, there are charts and diagrams in the report. The first chart is a pie chart with a red fill, representing a percentage distribution. The second chart is a bar chart with three bars representing different percentages: 100%, 75%, and 84%. The third chart is a red circular icon with three interconnected circles, likely representing a network or system of interconnected components.

SOURCES:
  - page 12 | visual
  - page 40 | visual
  - page 37 | visual
  - page 24 | visual
  - page 25 | visual


{'answer': 'Yes, there are charts and diagrams in the report. The first chart is a pie chart with a red fill, representing a percentage distribution. The second chart is a bar chart with three bars representing different percentages: 100%, 75%, and 84%. The third chart is a red circular icon with three interconnected circles, likely representing a network or system of interconnected components.',
 'hits': [{'text': 'The image is a pie chart with a red fill. The chart represents a percentage distribution, with the highest point at approximately 70% and the lowest point at around 30%. The chart shows a general trend of increasing values towards the right side, indicating a positive trend. The chart does not contain any specific numerical values or labels, but it visually represents the percentage distribution.',
   'image_path': 'extracted_images/page_12_img_1.png',
   'page': 12,
   'modality': 'visual'},
  {'text': 'The image contains a bar chart with key values and a main trend. The c

In [13]:
# 11. Export chroma_db + extracted_images for the Space
import shutil
import os

export_dir = Path('space_export')
export_dir.mkdir(exist_ok=True)
shutil.copytree(CHROMA_DIR, export_dir / 'chroma_db', dirs_exist_ok=True)
shutil.copytree(IMAGE_DIR, export_dir / 'extracted_images', dirs_exist_ok=True)

shutil.make_archive('space_export', 'zip', root_dir='.', base_dir='space_export')
print('Created space_export.zip — download it from the Kaggle output pane,')
print('then unzip its two folders into space/data/ in the project repo.')

Created space_export.zip — download it from the Kaggle output pane,
then unzip its two folders into space/data/ in the project repo.


In [14]:
# 12. Export index.json + flat images folder for the static Space
import json
import shutil

static_images_dir = Path('images_for_space')
static_images_dir.mkdir(exist_ok=True)

index_entries = []
for chunk, emb in zip(chunks, embeddings):
    image_filename = None
    if chunk.get('image_path'):
        src = Path(chunk['image_path'])
        image_filename = src.name
        shutil.copy(src, static_images_dir / image_filename)

    index_entries.append({
        'page': chunk['page'],
        'modality': chunk['modality'],
        'text': chunk['text'],
        'image': image_filename,
        'embedding': emb,
    })

with open('index.json', 'w') as f:
    json.dump(index_entries, f)

print(f'Wrote index.json with {len(index_entries)} entries')
print(f'Copied {len(list(static_images_dir.iterdir()))} images to images_for_space/')
print('Download both from the Kaggle output pane.')

Wrote index.json with 184 entries
Copied 90 images to images_for_space/
Download both from the Kaggle output pane.
